In [ ]:
# This project explores 370,000 datapoints of a German Ebay
# classified page, derived from Kaggle. I cleaned and performed an
# initial analysis on the dataset.
# source: https://www.kaggle.com/datasets/sijovm/used-cars-data-from-ebay-kleinanzeigen

import pandas as pd

url = 'https://github.com/nathanchapero-creator/eBay_used_car_analysis/raw/refs/heads/main/autos.csv.zip'

# Read the zipped CSV directly from the internet
autos = pd.read_csv(url, compression='zip', encoding='Latin-1')
autos_copy = autos
autos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371528 entries, 0 to 371527
Data columns (total 20 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   dateCrawled          371528 non-null  object
 1   name                 371528 non-null  object
 2   seller               371528 non-null  object
 3   offerType            371528 non-null  object
 4   price                371528 non-null  int64 
 5   abtest               371528 non-null  object
 6   vehicleType          333659 non-null  object
 7   yearOfRegistration   371528 non-null  int64 
 8   gearbox              351319 non-null  object
 9   powerPS              371528 non-null  int64 
 10  model                351044 non-null  object
 11  kilometer            371528 non-null  int64 
 12  monthOfRegistration  371528 non-null  int64 
 13  fuelType             338142 non-null  object
 14  brand                371528 non-null  object
 15  notRepairedDamage    299468 non-nu

In [ ]:
# renaming columns to snake_case
autos.rename({"yearOfRegistration":"registration_year", "monthOfRegistration":"registration_month","notRepairedDamage":"unrepaired_damage","dateCreated":"ad_created","power_p_s":"power_ps"}, axis = 1,inplace = True)
def camel_to_snake_case(name):
    new_name = ""
    for char in name:
        if char.isupper():
            new_name += "_" + char.lower()
        else:
            new_name += char.lower()
    return new_name

autos.columns = [camel_to_snake_case(col) for col in autos.columns]

## Column and Price Removals

Three columns, "seller", "offer_type", and "nr_of_pictures" are being removed due to nearly all of the listings having identical values, making them redundant for comparison.

Listings priced at $0 are removed due to most listings being placeholder listings, free car requests, or pricing errors.

Listings over $3.8 million removed to prevent outliers from skewing the overall average.

In [ ]:
# removing three columns
autos = autos.drop(["seller","offer_type","nr_of_pictures"], axis = 1)

#renaming "kilometer" column to "odometer_km"
autos.rename({"kilometer":"odometer_km"}, axis = 1, inplace = True)

# removing outlier prices over 3890000 and rows with no price (0)
autos = autos[(autos["price"] > 0) & (autos["price"] <= 3890000)]

## Mileage Distribution & Platform Constraints
Over 50% of the vehicles in this dataset report an exact odometer reading of 150,000 km. This indicates a hard cap built into the platform's user interface (likely a maximum drop-down value). Because of this artificial grouping, any car with more than 150k kilometers part of this final bucket, meaning graphs dependent on odometer values will most likely be inaccurate.

In [ ]:
# converting dates to datetime objects
autos['date_crawled'] = pd.to_datetime(autos["date_crawled"])
autos['ad_created'] = pd.to_datetime(autos["ad_created"])
autos['last_seen'] = pd.to_datetime(autos["last_seen"])

## Filtering Registration Years

I restricted the vehicle registration years to a window between 1950 and 2016.

Lower Bound (1950): Sample sizes drop off significantly for vehicles registered before 1950, and those that remain are highly likely to be user input errors rather than genuine antique car listings.

Upper Bound (2016): Because this dataset was scraped in 2016, any vehicle listing a registration year of 2017 or later is definitively an error. Removing these ensures the timeline remains accurate.

In [ ]:
# removing registration year outliers
autos = autos[autos["registration_year"].between(1950,2016)]

In [ ]:
## Goal: Select brands making up more than 2% of total listings, and calculate the average price and mileage (odometer_km) for each brand.

#Simple date check
matching_dates = autos[autos['date_crawled'].dt.normalize() == autos['ad_created'].dt.normalize()].shape[0]
quality_date_check = autos[autos['date_crawled'].dt.normalize() >= autos['ad_created'].dt.normalize()].shape[0]

autos['days_listed'] = (autos['last_seen'] - autos['ad_created']).dt.days
autos['car_age'] = (2016 - autos['registration_year'])

#replacing empty vehicle_type with "unknown"
autos["vehicle_type"] = autos["vehicle_type"].fillna("unknown")

# minor translation fixes for "vehicle_type"
autos["vehicle_type"] = autos["vehicle_type"].replace({
    "kleinwagen": "small car",
    "kombi": "station wagon",
    "cabrio": "convertible",
    "limousine": "sedan",
    "bus": "bus",
    "suv": "suv",
    "coupe": "coupe",
    "andere": "other"
})

## Removing Non-Sale "Junk" Listings

The dataset contains several entries that are not actual vehicles for sale, but rather want-ads, trade offers, or scrap sales. To ensure our price analysis reflects viable, functional cars, I filtered out listings containing specific German keywords in their titles, such as:

"suche" (searching/looking to buy)

"tausche" (looking to trade)

"ausschlachten" (dismantling/parting out for scrap)

In [ ]:
# checking for "junk listings"
junk_words = [
    'suche',
    'tausche',
    'tausch',
    'teile',
    'ausschlachten',
    'bastler'
]
junk_text = '|'.join(junk_words)

junk_listings = autos['name'].str.contains(junk_text, case = False, na = False)
junk_listings.value_counts()
# 10334 listings not "true sales"

# removing "junk listings" from autos
autos = autos[~junk_listings]

In [ ]:
print(autos.duplicated().sum())
# removing four duplicates
autos = autos.drop_duplicates()

4


In [ ]:
# dropping rows with a car age of 0
autos = autos[autos['car_age'] > 0]

# Exploratory Data Analysis:
### To conclude the initial cleaning phase, I wanted to quickly aggregate the top 10 brands (those making up more than 1.8% of the listings) to see if there were any immediate correlations between average price and average mileage. The odometer range being capped at 150,000 limited this analysis.

In [ ]:
# calculating top 10 most commonly listed brands
brand_counts = autos['brand'].value_counts(normalize=True)
selected_brands = brand_counts[brand_counts > 0.018].index

# filtering dataset to only include those brands
top_brands = autos[autos['brand'].isin(selected_brands)]

# grouping by brand, calculate mean, and assign to 'df'
top_brands = top_brands.groupby('brand')[['price', 'odometer_km']].mean().round(2)

# renaming columns
top_brands.columns = ['mean_price', 'mean_mileage']

# displaying final table
top_brands

,mean_price,mean_mileage
brand,,
audi,9361.53,129069.44
bmw,8675.57,132651.59
fiat,3018.44,115479.15
ford,3917.81,123306.98
mercedes_benz,8748.67,130343.64
opel,3119.06,128219.46
peugeot,3362.39,123921.32
renault,2543.88,127187.81
seat,4675.54,119908.99


In [ ]:
autos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 327514 entries, 0 to 371527
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date_crawled        327514 non-null  datetime64[ns]
 1   name                327514 non-null  object        
 2   price               327514 non-null  int64         
 3   abtest              327514 non-null  object        
 4   vehicle_type        327514 non-null  object        
 5   registration_year   327514 non-null  int64         
 6   gearbox             314301 non-null  object        
 7   power_p_s           327514 non-null  int64         
 8   model               313624 non-null  object        
 9   odometer_km         327514 non-null  int64         
 10  registration_month  327514 non-null  int64         
 11  fuel_type           308323 non-null  object        
 12  brand               327514 non-null  object        
 13  unrepaired_damage   274620 non-nul

In [ ]:
autos.to_csv('autos_cleaned.csv', index = False)
# files.download('autos_cleaned.csv')